In [29]:
import os
import json
from pathlib import Path
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import NumericType
from dotenv import load_dotenv


### MYSQL & Spark Configuration

In [30]:
PROJECT_ROOT = Path(
    r"D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics"
).resolve()
OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR", PROJECT_ROOT / "output"))
FINAL_PARQUET_PATH = str(OUTPUT_DIR / "final_schedule.parquet")
FEATURE_PARQUET_PATH = str(OUTPUT_DIR / "final_feature_dataset.parquet")
FEATURE_CSV_PATH = str(OUTPUT_DIR / "final_feature_dataset.csv")
FEATURE_METADATA_PATH = str(OUTPUT_DIR / "feature_metadata.json")
MYSQL_JAR = os.environ.get("MYSQL_JDBC_JAR", str(PROJECT_ROOT / "jdbc" / "mysql-connector-j-9.7.0.jar"))

N_CORES = int(os.environ.get("SPARK_CORES", "8"))


In [31]:
MYSQL_HOST = os.environ.get("MYSQL_HOST")
MYSQL_PORT = int(os.environ.get("MYSQL_PORT"))
MYSQL_USER = os.environ.get("MYSQL_USER")
MYSQL_PASSWORD = os.environ.get("MYSQL_PASSWORD")
MYSQL_DATABASE = os.environ.get("MYSQL_DATABASE")

assert MYSQL_PASSWORD, "MYSQL_PASSWORD is not set."

MYSQL_URI = (
    f"mysql+pymysql://{MYSQL_USER}:{quote_plus(MYSQL_PASSWORD)}@"
    f"{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}"
)

MYSQL_JDBC_URL = f"jdbc:mysql://{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}"

MYSQL_PROPERTIES = {
    "user": MYSQL_USER,
    "password": MYSQL_PASSWORD,
    "driver": "com.mysql.cj.jdbc.Driver",
}

print(f"MySQL target: {MYSQL_USER}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}")

MySQL target: root@localhost:3306/bus_performance_analytics


In [4]:
spark = (
    SparkSession.builder
    .appName("BusRoute_02_DataStorageProcessing")
    .master(f"local[{N_CORES}]")
    .config("spark.sql.shuffle.partitions", str(N_CORES * 2))
    .config("spark.driver.memory", os.environ.get("SPARK_DRIVER_MEMORY", "4g"))
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.autoBroadcastJoinThreshold", -1)
    .config("spark.jars", MYSQL_JAR)
    .config("spark.driver.extraClassPath", MYSQL_JAR)
    .config("spark.executor.extraClassPath", MYSQL_JAR)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("Spark UI     :", spark.sparkContext.uiWebUrl)

Spark version: 3.5.8
Spark UI     : http://DESKTOP-P7PE4MO:4040


In [5]:
schedule = spark.read.parquet(FINAL_PARQUET_PATH).cache()
BASE_ROW_COUNT = schedule.count()

print("="*60)
print("Dataset Loaded")
print("="*60)
print("Rows    :", f"{BASE_ROW_COUNT:,}")
print("Columns :", len(schedule.columns))
schedule.printSchema()

assert BASE_ROW_COUNT > 0, "final_schedule.parquet loaded with zero rows."
natural_key = ["source_file", "vehicle_journey_code", "stop_point_ref", "stop_sequence"]
assert schedule.dropDuplicates(natural_key).count() == BASE_ROW_COUNT, \
    "Duplicate rows on the natural key -- "

Dataset Loaded
Rows    : 926,481
Columns : 16
root
 |-- line_ref: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- stop_point_ref: string (nullable = true)
 |-- vehicle_journey_code: string (nullable = true)
 |-- stop_sequence: integer (nullable = true)
 |-- scheduled_time: string (nullable = true)
 |-- scheduled_ts: timestamp (nullable = true)
 |-- service_ref: string (nullable = true)
 |-- line_name: string (nullable = true)
 |-- operator_ref: string (nullable = true)
 |-- journey_pattern_ref: string (nullable = true)
 |-- scheduled_departure_time: string (nullable = true)
 |-- scheduled_departure_ts: timestamp (nullable = true)
 |-- stop_name: string (nullable = true)
 |-- fare_publication_count: long (nullable = true)
 |-- has_fare_data: integer (nullable = true)



In [6]:
def find_constant_or_null_columns(df, exclude=()):
    '''Returns {column: reason} for any column that is 100% null or has at
    most one distinct non-null value (i.e. carries no information).'''
    cols = [c for c in df.columns if c not in exclude]
    n = df.count()
    stats = df.select([
        F.countDistinct(F.col(c)).alias(f"{c}__distinct") for c in cols
    ] + [
        F.sum(F.col(c).isNull().cast("int")).alias(f"{c}__nulls") for c in cols
    ]).collect()[0].asDict()

    flagged = {}
    for c in cols:
        n_distinct = stats[f"{c}__distinct"]
        n_nulls = stats[f"{c}__nulls"]
        if n_nulls == n:
            flagged[c] = "100% null"
        elif n_distinct <= 1:
            flagged[c] = f"constant (single value, {n_distinct} distinct non-null)"
    return flagged


def find_duplicate_columns(df, columns=None):
    '''Cheap fingerprint pass (crc32 sum + distinct count + null count per
    column) to find *candidate* duplicate column pairs, then an exact
    row-for-row equality check only on those candidates -- avoids an O(n^2)
    full-equality scan across every column pair on a 900k-row table.'''
    columns = columns or df.columns
    fp = df.select(
        [F.sum(F.crc32(F.col(c).cast("string"))).alias(f"{c}__crc") for c in columns]
        + [F.countDistinct(F.col(c)).alias(f"{c}__nd") for c in columns]
        + [F.sum(F.col(c).isNull().cast("int")).alias(f"{c}__null") for c in columns]
    ).collect()[0].asDict()

    groups = {}
    for c in columns:
        key = (fp[f"{c}__crc"], fp[f"{c}__nd"], fp[f"{c}__null"])
        groups.setdefault(key, []).append(c)

    confirmed_duplicates = []
    for candidates in groups.values():
        if len(candidates) < 2:
            continue
        for i in range(len(candidates)):
            for j in range(i + 1, len(candidates)):
                a, b = candidates[i], candidates[j]
                mismatches = df.filter(~F.col(a).eqNullSafe(F.col(b))).count()
                if mismatches == 0:
                    confirmed_duplicates.append((a, b))
    return confirmed_duplicates


print("Column-quality gate functions ready.")

Column-quality gate functions ready.


In [7]:
FARE_PROTECTED_COLUMNS = ("has_fare_data", "fare_publication_count")

constant_cols = find_constant_or_null_columns(schedule, exclude=FARE_PROTECTED_COLUMNS)
duplicate_col_pairs = find_duplicate_columns(
    schedule, columns=[c for c in schedule.columns if c not in FARE_PROTECTED_COLUMNS]
)

print("Constant / all-null columns in base data:", constant_cols or "none")
print("Duplicate column pairs in base data:", duplicate_col_pairs or "none")

for col_name in constant_cols:
    schedule = schedule.drop(col_name)
for a, b in duplicate_col_pairs:
    if b in schedule.columns:
        schedule = schedule.drop(b)

print(f"\nColumns after base-data gate: {len(schedule.columns)}")

Constant / all-null columns in base data: none
Duplicate column pairs in base data: [('scheduled_time', 'scheduled_departure_time'), ('scheduled_ts', 'scheduled_departure_ts')]

Columns after base-data gate: 14


## Time-based features

In [8]:
schedule = (
    schedule
    .withColumn("hour_of_day", F.hour("scheduled_ts"))
    .withColumn("day_of_week", F.dayofweek("scheduled_ts"))
    .withColumn("month", F.month("scheduled_ts"))
)
schedule = schedule.withColumn(
    "is_weekend", F.when(F.col("day_of_week").isin([1, 7]), 1).otherwise(0)
)
schedule = schedule.withColumn(
    "is_peak_hour",
    F.when(
        F.col("hour_of_day").between(7, 9) | F.col("hour_of_day").between(16, 18), 1
    ).otherwise(0)
)

schedule.select(
    "scheduled_ts", "hour_of_day", "day_of_week", "month", "is_weekend", "is_peak_hour"
).show(10, truncate=False)

n_null_hour = schedule.filter(F.col("hour_of_day").isNull()).count()
print(f"\nRows with unparseable hour_of_day: {n_null_hour:,} ({100*n_null_hour/BASE_ROW_COUNT:.2f}%)")
assert n_null_hour / BASE_ROW_COUNT < 0.01, \
    "hour_of_day is null for more than 1% of rows -- scheduled_ts parsing needs investigating before trusting time features."
assert schedule.count() == BASE_ROW_COUNT

+-------------------+-----------+-----------+-----+----------+------------+
|scheduled_ts       |hour_of_day|day_of_week|month|is_weekend|is_peak_hour|
+-------------------+-----------+-----------+-----+----------+------------+
|1970-01-01 15:32:00|15         |5          |1    |0         |0           |
|1970-01-01 04:05:00|4          |5          |1    |0         |0           |
|1970-01-01 22:10:00|22         |5          |1    |0         |0           |
|1970-01-01 08:55:00|8          |5          |1    |0         |1           |
|1970-01-01 07:05:00|7          |5          |1    |0         |1           |
|1970-01-01 11:32:00|11         |5          |1    |0         |0           |
|1970-01-01 07:30:00|7          |5          |1    |0         |1           |
|1970-01-01 17:25:00|17         |5          |1    |0         |1           |
|1970-01-01 16:30:00|16         |5          |1    |0         |1           |
|1970-01-01 14:45:00|14         |5          |1    |0         |0           |
+-----------

## Journey-based features

In [9]:
total_stops = (
    schedule.groupBy("source_file", "vehicle_journey_code")
    .agg((F.max("stop_sequence") + 1).alias("total_stops"))
)
schedule = schedule.join(F.broadcast(total_stops), ["source_file", "vehicle_journey_code"], "left")

print("Rows:", schedule.count())
assert schedule.count() == BASE_ROW_COUNT
schedule.select("vehicle_journey_code", "total_stops").show(5, truncate=False)

Rows: 926481
+--------------------+-----------+
|vehicle_journey_code|total_stops|
+--------------------+-----------+
|VJ752               |22         |
|VJ440               |34         |
|VJ100               |37         |
|VJ558               |44         |
|VJ1336              |31         |
+--------------------+-----------+
only showing top 5 rows



In [10]:
journey_window = Window.partitionBy("source_file", "vehicle_journey_code")

schedule = (
    schedule
    .withColumn("journey_start", F.min("scheduled_ts").over(journey_window))
    .withColumn("journey_end", F.max("scheduled_ts").over(journey_window))
)

raw_duration_s = F.unix_timestamp("journey_end") - F.unix_timestamp("journey_start")
n_overnight = schedule.filter(raw_duration_s < 0).count()
print(f"Journeys with apparent negative duration (likely overnight wrap): {n_overnight:,}")

schedule = schedule.withColumn(
    "journey_duration_minutes",
    F.round(F.when(raw_duration_s < 0, raw_duration_s + 24 * 3600).otherwise(raw_duration_s) / 60, 2)
)

print("Rows:", schedule.count())
assert schedule.count() == BASE_ROW_COUNT
schedule.select("vehicle_journey_code", "journey_start", "journey_end", "journey_duration_minutes").show(10, truncate=False)
schedule.select(
    F.min("journey_duration_minutes").alias("min"),
    F.max("journey_duration_minutes").alias("max"),
    F.mean("journey_duration_minutes").alias("mean"),
).show()

Journeys with apparent negative duration (likely overnight wrap): 0
Rows: 926481
+--------------------+-------------------+-------------------+------------------------+
|vehicle_journey_code|journey_start      |journey_end        |journey_duration_minutes|
+--------------------+-------------------+-------------------+------------------------+
|VJ482               |1970-01-01 07:19:00|1970-01-01 07:19:00|0.0                     |
|VJ482               |1970-01-01 07:19:00|1970-01-01 07:19:00|0.0                     |
|VJ482               |1970-01-01 07:19:00|1970-01-01 07:19:00|0.0                     |
|VJ482               |1970-01-01 07:19:00|1970-01-01 07:19:00|0.0                     |
|VJ482               |1970-01-01 07:19:00|1970-01-01 07:19:00|0.0                     |
|VJ482               |1970-01-01 07:19:00|1970-01-01 07:19:00|0.0                     |
|VJ482               |1970-01-01 07:19:00|1970-01-01 07:19:00|0.0                     |
|VJ482               |1970-01-01 07:19:

In [11]:
schedule = schedule.withColumn(
    "stop_progress_pct",
    F.round(
        F.when(F.col("total_stops") > 1, (F.col("stop_sequence") / (F.col("total_stops") - 1)) * 100)
         .otherwise(0),
        2,
    ),
)

# Data-driven tertile thresholds instead of arbitrary fixed numbers.
q1, q2 = schedule.approxQuantile("total_stops", [0.33, 0.66], 0.01)
print(f"total_stops tertile thresholds: {q1:.1f} / {q2:.1f}")

schedule = schedule.withColumn(
    "journey_type",
    F.when(F.col("total_stops") >= q2, "Long")
     .when(F.col("total_stops") >= q1, "Medium")
     .otherwise("Short"),
)

schedule = schedule.drop("journey_start", "journey_end")

print("Rows:", schedule.count())
assert schedule.count() == BASE_ROW_COUNT
schedule.groupBy("journey_type").count().show()

total_stops tertile thresholds: 31.0 / 46.0
Rows: 926481
+------------+------+
|journey_type| count|
+------------+------+
|       Short|276677|
|      Medium|318522|
|        Long|331282|
+------------+------+



## Route-based features

In [12]:
route_daily = schedule.groupBy("line_ref").agg(F.countDistinct("vehicle_journey_code").alias("daily_journeys"))
schedule = schedule.join(F.broadcast(route_daily), on="line_ref", how="left")

route_stops = schedule.groupBy("line_ref").agg(F.countDistinct("stop_point_ref").alias("unique_stops"))
schedule = schedule.join(F.broadcast(route_stops), on="line_ref", how="left")

route_duration = schedule.groupBy("line_ref").agg(F.avg("journey_duration_minutes").alias("route_average_duration"))
schedule = schedule.join(F.broadcast(route_duration), on="line_ref", how="left")

print("Rows:", schedule.count())
assert schedule.count() == BASE_ROW_COUNT

Rows: 926481


In [13]:
dj_q1, dj_q2 = schedule.approxQuantile("daily_journeys", [0.33, 0.66], 0.01)
us_q1, us_q2 = schedule.approxQuantile("unique_stops", [0.33, 0.66], 0.01)
print(f"daily_journeys tertile thresholds: {dj_q1:.1f} / {dj_q2:.1f}")
print(f"unique_stops tertile thresholds:  {us_q1:.1f} / {us_q2:.1f}")

schedule = schedule.withColumn(
    "route_popularity",
    F.when(F.col("daily_journeys") >= dj_q2, "High")
     .when(F.col("daily_journeys") >= dj_q1, "Medium")
     .otherwise("Low"),
)
schedule = schedule.withColumn(
    "route_complexity",
    F.when(F.col("unique_stops") >= us_q2, "High")
     .when(F.col("unique_stops") >= us_q1, "Medium")
     .otherwise("Low"),
)

print("Rows:", schedule.count())
assert schedule.count() == BASE_ROW_COUNT
schedule.select("line_ref", "daily_journeys", "unique_stops", "route_average_duration",
                "route_popularity", "route_complexity").show(10, truncate=False)
schedule.groupBy("route_popularity").count().orderBy(F.desc("count")).show()

daily_journeys tertile thresholds: 170.0 / 309.0
unique_stops tertile thresholds:  62.0 / 93.0
Rows: 926481
+----------------------+--------------+------------+----------------------+----------------+----------------+
|line_ref              |daily_journeys|unique_stops|route_average_duration|route_popularity|route_complexity|
+----------------------+--------------+------------+----------------------+----------------+----------------+
|SCOX:PH0005863:91:B9  |272           |50          |0.0                   |Medium          |Low             |
|SCGL:PH0005031:188:94 |400           |70          |0.0                   |High            |Medium          |
|SCGL:PH0005031:188:94 |400           |70          |0.0                   |High            |Medium          |
|SCGL:PH0005031:290:43A|40            |70          |0.0                   |Low             |Medium          |
|SCGL:PH0005031:180:A  |600           |71          |0.0                   |High            |Medium          |
|SCCM:PF0000

## Stop-based features

In [14]:
schedule = schedule.withColumn("is_first_stop", F.when(F.col("stop_sequence") == 1, 1).otherwise(0))
schedule = schedule.withColumn(
    "is_last_stop", F.when(F.col("stop_sequence") == F.col("total_stops"), 1).otherwise(0)
)
schedule = schedule.withColumn(
    "stop_position",
    F.when(F.col("stop_progress_pct") <= 25, "Beginning")
     .when(F.col("stop_progress_pct") <= 75, "Middle")
     .otherwise("End"),
)

stop_activity = schedule.groupBy("stop_point_ref").agg(F.count("*").alias("stop_activity"))
schedule = schedule.join(F.broadcast(stop_activity), on="stop_point_ref", how="left")

sa_q1, sa_q2 = schedule.approxQuantile("stop_activity", [0.33, 0.66], 0.01)
schedule = schedule.withColumn(
    "stop_busyness",
    F.when(F.col("stop_activity") >= sa_q2, "High")
     .when(F.col("stop_activity") >= sa_q1, "Medium")
     .otherwise("Low"),
)

print("Rows:", schedule.count())
assert schedule.count() == BASE_ROW_COUNT
schedule.select("stop_point_ref", "stop_sequence", "total_stops", "stop_progress_pct",
                "is_first_stop", "is_last_stop", "stop_position", "stop_activity", "stop_busyness"
                ).show(10, truncate=False)

Rows: 926481
+--------------+-------------+-----------+-----------------+-------------+------------+-------------+-------------+-------------+
|stop_point_ref|stop_sequence|total_stops|stop_progress_pct|is_first_stop|is_last_stop|stop_position|stop_activity|stop_busyness|
+--------------+-------------+-----------+-----------------+-------------+------------+-------------+-------------+-------------+
|340001458HOR  |18           |22         |85.71            |0            |0           |End          |345          |High         |
|1600GLA642    |4            |34         |12.12            |0            |0           |Beginning    |282          |Medium       |
|1600GL4429    |14           |37         |38.89            |0            |0           |Middle       |282          |Medium       |
|1600GLT108    |11           |44         |25.58            |0            |0           |Middle       |47           |Low          |
|1600GLA36586  |5            |31         |16.67            |0            |0  

## Operator-based and fare-based features

In [15]:
operator_routes = schedule.groupBy("operator_ref").agg(F.countDistinct("line_ref").alias("operator_routes"))
schedule = schedule.join(F.broadcast(operator_routes), on="operator_ref", how="left")

operator_trips = schedule.groupBy("operator_ref").agg(F.countDistinct("vehicle_journey_code").alias("operator_trip_count"))
schedule = schedule.join(F.broadcast(operator_trips), on="operator_ref", how="left")

operator_duration = schedule.groupBy("operator_ref").agg(F.avg("journey_duration_minutes").alias("operator_avg_duration"))
schedule = schedule.join(F.broadcast(operator_duration), on="operator_ref", how="left")

or_q1, or_q2 = schedule.approxQuantile("operator_routes", [0.33, 0.66], 0.01)
ot_q1, ot_q2 = schedule.approxQuantile("operator_trip_count", [0.33, 0.66], 0.01)

schedule = schedule.withColumn(
    "operator_size",
    F.when(F.col("operator_routes") >= or_q2, "Large")
     .when(F.col("operator_routes") >= or_q1, "Medium")
     .otherwise("Small"),
)
schedule = schedule.withColumn(
    "operator_workload",
    F.when(F.col("operator_trip_count") >= ot_q2, "High")
     .when(F.col("operator_trip_count") >= ot_q1, "Medium")
     .otherwise("Low"),
)

print("Rows:", schedule.count())
assert schedule.count() == BASE_ROW_COUNT
schedule.select("operator_ref", "operator_routes", "operator_trip_count", "operator_avg_duration",
                "operator_size", "operator_workload").show(10, truncate=False)

Rows: 926481
+------------+---------------+-------------------+---------------------+-------------+-----------------+
|operator_ref|operator_routes|operator_trip_count|operator_avg_duration|operator_size|operator_workload|
+------------+---------------+-------------------+---------------------+-------------+-----------------+
|11          |42             |3968               |0.0                  |Medium       |High             |
|1           |113            |3434               |0.0                  |Large        |High             |
|1           |113            |3434               |0.0                  |Large        |High             |
|1           |113            |3434               |0.0                  |Large        |High             |
|1           |113            |3434               |0.0                  |Large        |High             |
|1           |113            |3434               |0.0                  |Large        |High             |
|1           |113            |3434        

In [16]:
fpl_q1, fpl_q2 = schedule.approxQuantile("fare_publication_count", [0.33, 0.66], 0.01)

schedule = schedule.withColumn(
    "fare_status", F.when(F.col("has_fare_data") == 1, "Available").otherwise("Unavailable")
)
schedule = schedule.withColumn(
    "fare_publication_level",
    F.when(F.col("fare_publication_count") >= fpl_q2, "High")
     .when(F.col("fare_publication_count") >= fpl_q1, "Medium")
     .otherwise("Low"),
)
schedule = schedule.withColumn(
    "fare_complexity_score", F.col("fare_publication_count") * F.col("has_fare_data")
)
print("Rows:", schedule.count())
assert schedule.count() == BASE_ROW_COUNT
schedule.select("line_ref", "fare_publication_count", "has_fare_data", "fare_status",
                "fare_publication_level", "fare_complexity_score"
                ).show(10, truncate=False)

Rows: 926481
+----------------------+----------------------+-------------+-----------+----------------------+---------------------+
|line_ref              |fare_publication_count|has_fare_data|fare_status|fare_publication_level|fare_complexity_score|
+----------------------+----------------------+-------------+-----------+----------------------+---------------------+
|SCOX:PH0005863:91:B9  |1                     |1            |Available  |High                  |1                    |
|SCGL:PH0005031:188:94 |1                     |1            |Available  |High                  |1                    |
|SCGL:PH0005031:188:94 |1                     |1            |Available  |High                  |1                    |
|SCGL:PH0005031:290:43A|1                     |1            |Available  |High                  |1                    |
|SCGL:PH0005031:180:A  |1                     |1            |Available  |High                  |1                    |
|SCCM:PF0000459:218:PR5|0          

In [17]:
constant_cols_final = find_constant_or_null_columns(schedule)
duplicate_col_pairs_final = find_duplicate_columns(
    schedule, columns=[c for c in schedule.columns if c not in ("scheduled_ts", "scheduled_departure_ts")]
)

print("Constant / all-null columns found after feature engineering:")
for c, reason in constant_cols_final.items():
    print(f"  {c}: {reason}")
print("\nDuplicate column pairs found:", duplicate_col_pairs_final or "none")

for col_name in constant_cols_final:
    schedule = schedule.drop(col_name)
for a, b in duplicate_col_pairs_final:
    if b in schedule.columns:
        schedule = schedule.drop(b)

print(f"\nColumns remaining after final gate: {len(schedule.columns)}")
print(schedule.columns)

Constant / all-null columns found after feature engineering:
  day_of_week: constant (single value, 1 distinct non-null)
  month: constant (single value, 1 distinct non-null)
  is_weekend: constant (single value, 1 distinct non-null)
  journey_duration_minutes: constant (single value, 1 distinct non-null)
  route_average_duration: constant (single value, 1 distinct non-null)
  is_last_stop: constant (single value, 1 distinct non-null)
  operator_avg_duration: constant (single value, 1 distinct non-null)

Duplicate column pairs found: [('fare_publication_count', 'has_fare_data'), ('fare_publication_count', 'fare_complexity_score'), ('has_fare_data', 'fare_complexity_score'), ('is_weekend', 'is_last_stop'), ('journey_duration_minutes', 'route_average_duration'), ('journey_duration_minutes', 'operator_avg_duration'), ('route_average_duration', 'operator_avg_duration')]

Columns remaining after final gate: 32
['operator_ref', 'stop_point_ref', 'line_ref', 'source_file', 'vehicle_journey_co

## Feature-role manifest

In [18]:
TARGET_COLUMN = "route_popularity"
LEAKAGE_COLUMNS = ["daily_journeys"]

ID_COLUMNS = [
    "source_file", "vehicle_journey_code", "stop_point_ref", "line_ref",
    "line_name", "service_ref", "journey_pattern_ref", "stop_name",
    "operator_ref", 
]

TIMESTAMP_COLUMNS = [c for c in ["scheduled_ts", "scheduled_departure_ts"] if c in schedule.columns]

all_cols = set(schedule.columns)
reserved = set(ID_COLUMNS) | set(TIMESTAMP_COLUMNS) | set(LEAKAGE_COLUMNS) | {TARGET_COLUMN}
candidate_feature_columns = sorted(all_cols - reserved)

assert "service_category" not in schedule.columns, (
    "service_category is present but is target-derived leakage (built from "
    "route_popularity) -- it must be removed at the point of creation, not "
    "just excluded from candidate_feature_columns."
)

feature_metadata = {
    "target_column": TARGET_COLUMN,
    "leakage_columns": LEAKAGE_COLUMNS,
    "removed_leakage_features_at_source": ["service_category"],
    "id_columns": ID_COLUMNS,
    "timestamp_columns": TIMESTAMP_COLUMNS,
    "candidate_feature_columns": candidate_feature_columns,
    "dropped_constant_columns": {**constant_cols, **constant_cols_final},
    "dropped_duplicate_columns": [b for _, b in (duplicate_col_pairs + duplicate_col_pairs_final)],
}

with open(FEATURE_METADATA_PATH, "w") as f:
    json.dump(feature_metadata, f, indent=2)

## Final validation and save

In [19]:
print("="*70)
print("FINAL FEATURE ENGINEERING SUMMARY")
print("="*70)
final_rows = schedule.count()
print(f"Rows    : {final_rows:,}")
print(f"Columns : {len(schedule.columns)}")
assert final_rows == BASE_ROW_COUNT, "Row count drifted during feature engineering -- investigate before saving."

null_counts = schedule.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in schedule.columns
]).collect()[0].asDict()
non_zero_nulls = {c: n for c, n in null_counts.items() if n > 0}
print("\nColumns with remaining nulls:", non_zero_nulls or "none")

schedule.printSchema()

FINAL FEATURE ENGINEERING SUMMARY
Rows    : 926,481
Columns : 32

Columns with remaining nulls: {'stop_name': 414980}
root
 |-- operator_ref: string (nullable = true)
 |-- stop_point_ref: string (nullable = true)
 |-- line_ref: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- vehicle_journey_code: string (nullable = true)
 |-- stop_sequence: integer (nullable = true)
 |-- scheduled_time: string (nullable = true)
 |-- scheduled_ts: timestamp (nullable = true)
 |-- service_ref: string (nullable = true)
 |-- line_name: string (nullable = true)
 |-- journey_pattern_ref: string (nullable = true)
 |-- stop_name: string (nullable = true)
 |-- fare_publication_count: long (nullable = true)
 |-- hour_of_day: integer (nullable = true)
 |-- is_peak_hour: integer (nullable = false)
 |-- total_stops: integer (nullable = true)
 |-- stop_progress_pct: double (nullable = true)
 |-- journey_type: string (nullable = false)
 |-- daily_journeys: long (nullable = true)
 |-- unique_s

### Cell 18 — Save engineered dataset (Parquet + CSV)

In [20]:
schedule.repartition(N_CORES).write.mode("overwrite").parquet(FEATURE_PARQUET_PATH)
print(f"Parquet saved to: {FEATURE_PARQUET_PATH}")

verify = spark.read.parquet(FEATURE_PARQUET_PATH)
print(f"Verify -- rows: {verify.count():,}, columns: {len(verify.columns)}")

Parquet saved to: D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics\output\final_feature_dataset.parquet
Verify -- rows: 926,481, columns: 32


In [21]:
import glob, shutil

csv_folder = str(Path(FEATURE_CSV_PATH).with_suffix("")) + "_tmp"
(
    verify.coalesce(1).write
    .mode("overwrite")
    .option("header", True)
    .csv(csv_folder)
)
part_file = glob.glob(str(Path(csv_folder) / "part-*.csv"))[0]
if Path(FEATURE_CSV_PATH).exists():
    Path(FEATURE_CSV_PATH).unlink()
shutil.move(part_file, FEATURE_CSV_PATH)
shutil.rmtree(csv_folder)
print(f"CSV saved to: {FEATURE_CSV_PATH}")

CSV saved to: D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics\output\final_feature_dataset.csv


### MySQL persistence

In [22]:

temp_df = schedule
for field in temp_df.schema.fields:
    if field.dataType.typeName() == "timestamp":
        temp_df = temp_df.withColumn(field.name, F.date_format(F.col(field.name), "yyyy-MM-dd HH:mm:ss"))

(
    temp_df.write.mode("overwrite")
    .jdbc(url=MYSQL_JDBC_URL, table="feature_engineered_schedule", properties=MYSQL_PROPERTIES)
)
print("Feature-engineered dataset saved to MySQL.")

check = spark.read.jdbc(url=MYSQL_JDBC_URL, table="feature_engineered_schedule", properties=MYSQL_PROPERTIES)
print("MySQL rows:", f"{check.count():,}")

Feature-engineered dataset saved to MySQL.
MySQL rows: 926,481


In [23]:
spark.stop()
print("Spark session stopped. Notebook 04 complete.")

Spark session stopped. Notebook 04 complete.
